# 06 — One RAG Query, End to End

This notebook follows one question through every essential implementation step:

```text
Question → query embedding → cosine ranking → Top-3 chunks
         → evidence gate → prompt → Qwen chat template
         → model.generate() → decoded answer → retrieved-source list
```

The cells deliberately show the core code directly rather than hiding the complete flow inside `run_rag()`.

## 0 — Load the two models and saved retrieval artifacts

MiniLM is the **retriever model**; Qwen is the **generator model**. The saved matrix contains one 384-dimensional MiniLM vector for each of the 972 chunks.

In [1]:
from pathlib import Path
import os, sys, logging
import numpy as np
import pandas as pd
import torch

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger('tensorflow').setLevel(logging.ERROR)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.embed_retrieve import read_jsonl, load_embedding_model
from src.rag_generate import (
    load_generator, build_prompt, unique_retrieved_sources,
    format_retrieved_source_list
)

embedding_model = load_embedding_model(PROJECT_ROOT / 'models/all-MiniLM-L6-v2')
chunks = read_jsonl(PROJECT_ROOT / 'data/processed/chunks_128.jsonl')
chunk_embeddings = np.load(PROJECT_ROOT / 'data/embeddings/embeddings_128.npy')
generator_tokenizer, generator_model = load_generator(
    PROJECT_ROOT / 'models/qwen2.5-0.5b-instruct'
)

print('Chunk records:', len(chunks))
print('Embedding matrix:', chunk_embeddings.shape)
print('Generator device:', generator_model.device)

Chunk records: 972
Embedding matrix: (972, 384)
Generator device: cpu


## 1 — Receive the user question

In [2]:
question = (
    'What safety and maintainability benefits are associated with '
    'prognostics and health management?'
)
print(question)

What safety and maintainability benefits are associated with prognostics and health management?


## 2 — Convert the question into a MiniLM vector

`normalize_embeddings=True` makes the query vector length equal to 1. The chunk vectors were generated with the same model and normalization.

In [3]:
query_vector = embedding_model.encode(
    [question],
    convert_to_numpy=True,
    normalize_embeddings=True,
)[0]

print('Query-vector shape:', query_vector.shape)
print('Query-vector norm:', round(float(np.linalg.norm(query_vector)), 6))
print('First 8 vector values:', np.round(query_vector[:8], 4))

Query-vector shape: (384,)
Query-vector norm: 1.0
First 8 vector values: [-0.0249  0.0378 -0.0472  0.0382  0.082   0.0569 -0.0041  0.0803]


## 3 — Rank all chunks with cosine similarity

Because query and chunk vectors are normalized, matrix multiplication calculates all 972 cosine similarities in one operation. `argsort()[::-1]` changes ascending indexes into highest-first ranking.

In [4]:
# Essential semantic-search implementation
scores = chunk_embeddings @ query_vector
top_k = 3
top_indices = np.argsort(scores)[::-1][:top_k]

retrieved = []
for rank, index in enumerate(top_indices, start=1):
    chunk = chunks[int(index)]
    retrieved.append({
        'rank': rank,
        'score': float(scores[index]),
        'chunk_id': chunk['chunk_id'],
        'document': chunk['document'],
        'pdf_page': chunk['pdf_page'],
        'text': chunk['text'],
    })

pd.DataFrame([{
    'rank': r['rank'], 'cosine_similarity': r['score'],
    'document': r['document'], 'pdf_page': r['pdf_page'],
    'chunk_id': r['chunk_id'], 'preview': r['text'][:180]
} for r in retrieved]).style.format({'cosine_similarity': '{:.3f}'})

,rank,cosine_similarity,document,pdf_page,chunk_id,preview
0,1,0.806,NIST.AMS.100-18,17,NIST.AMS.100-18-p017-t128-c001,"Bo et al identify a number of benefits of prognostics and health management, a component related to predictive maintenance, which include34: • Safety: Advance warning of failure an"
1,2,0.689,NIST.AMS.100-2,7,NIST.AMS.100-2-p007-t128-c007,3 1.2. Background and Importance of Prognostics and Health Management .......................................... 3 1.3.
2,3,0.657,NIST.AMS.100-2,10,NIST.AMS.100-2-p010-t128-c008,Figure 1-1. General Prognostics and Health Management Design Components to Detect and Mitigate Equipment Failure


## 4 — Apply the evidence gate

A retriever always returns a nearest result. The selected grounded MVP checks the highest similarity before calling Qwen. The current `0.58` threshold was calibrated on the small project evaluation set and is not a probability.

In [5]:
minimum_evidence_score = 0.58
top_similarity = retrieved[0]['score']
evidence_passed = top_similarity >= minimum_evidence_score

print('Top similarity:', round(top_similarity, 3))
print('Required minimum:', minimum_evidence_score)
print('Evidence passed:', evidence_passed)

if not evidence_passed:
    raise RuntimeError('INSUFFICIENT EVIDENCE — Qwen should not be called')

Top similarity: 0.806
Required minimum: 0.58
Evidence passed: True


## 5 — Build the grounded prompt within the generator budget

`build_prompt()` adds complete retrieved chunks in rank order. It tokenizes the entire chat-formatted candidate before accepting another chunk, preventing silent evidence truncation.

In [6]:
prompt, selected_sources, input_token_count = build_prompt(
    question,
    retrieved,
    generator_tokenizer,
    style='grounded',
    max_input_tokens=1024,
)

print('Retrieved Top-k:', len(retrieved))
print('Complete chunks placed in prompt:', len(selected_sources))
print('Chat-formatted input tokens:', input_token_count)
print('\nRAW RAG PROMPT\n', '-' * 100)
print(prompt)

Retrieved Top-k: 3
Complete chunks placed in prompt: 3
Chat-formatted input tokens: 321

RAW RAG PROMPT
 ----------------------------------------------------------------------------------------------------
Answer using only the evidence below in no more than three concise sentences. State the answer directly, do not discuss these instructions, and do not use outside knowledge. Source citations are attached separately by the system.

EVIDENCE:
[S1 | NIST.AMS.100-18, PDF p.17]
Bo et al identify a number of benefits of prognostics and health management, a component related to predictive maintenance, which include34:
• Safety: Advance warning of failure and avoiding a catastrophic failure
• Maintainability: Eliminating redundant inspections, minimizing unscheduled maintenance, and decreasing test equipment requirement
• Logistics: Improving and assisting in the design of logistical support system
• Life-cycle costs: reducing operational and support costs
• System design and analysis: Impro

## 6 — Apply Qwen's chat template

Our RAG prompt is the message content. The tokenizer adds Qwen-specific user/assistant control tokens. `add_generation_prompt=True` ends at the point where the assistant should begin producing text.

In [7]:
messages = [
    {'role': 'user', 'content': prompt}
]

chat_formatted_text = generator_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat_formatted_text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Answer using only the evidence below in no more than three concise sentences. State the answer directly, do not discuss these instructions, and do not use outside knowledge. Source citations are attached separately by the system.

EVIDENCE:
[S1 | NIST.AMS.100-18, PDF p.17]
Bo et al identify a number of benefits of prognostics and health management, a component related to predictive maintenance, which include34:
• Safety: Advance warning of failure and avoiding a catastrophic failure
• Maintainability: Eliminating redundant inspections, minimizing unscheduled maintenance, and decreasing test equipment requirement
• Logistics: Improving and assisting in the design of logistical support system
• Life-cycle costs: reducing operational and support costs
• System design and analysis: Improving design and qualifications along with improving reliability prediction accuracy
• Relia

## 7 — Tokenize and call `model.generate()`

This is the direct generator call. `do_sample=False` uses deterministic greedy decoding. The returned tensor contains both the input tokens and newly generated tokens, so the input prefix is removed before decoding the answer.

In [8]:
# Convert chat-formatted messages directly into model token IDs
input_ids = generator_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to(generator_model.device)
attention_mask = torch.ones_like(input_ids)

# Essential Hugging Face LLM generation implementation
with torch.inference_mode():
    output_ids = generator_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=96,
        do_sample=False,
        num_beams=1,
        pad_token_id=generator_tokenizer.eos_token_id,
        temperature=None,
        top_p=None,
        top_k=None,
    )

# For a causal LLM, remove the original prompt tokens before decoding
generated_token_ids = output_ids[0, input_ids.shape[1]:]
raw_model_answer = generator_tokenizer.decode(
    generated_token_ids,
    skip_special_tokens=True,
).strip()

print('Input tensor shape:', tuple(input_ids.shape))
print('New tokens generated:', len(generated_token_ids))
print('\nRAW MODEL ANSWER\n', '-' * 100)
print(raw_model_answer)

Input tensor shape: (1, 321)
New tokens generated: 36

RAW MODEL ANSWER
 ----------------------------------------------------------------------------------------------------
Safety and maintainability benefits associated with prognostics and health management include advanced warning of failure, elimination of redundant inspections, minimized unscheduled maintenance, and decreased test equipment requirement.


## 8 — Append deterministic retrieved-source metadata

The model is not asked to invent or reproduce citations. The application deduplicates the document/pages of every complete chunk actually supplied in the prompt and appends them after the generated text.

In [9]:
retrieved_source_metadata = unique_retrieved_sources(selected_sources)
source_list = format_retrieved_source_list(retrieved_source_metadata)
final_output = f'{raw_model_answer}\n\n{source_list}'

print('FINAL APPLICATION OUTPUT\n', '=' * 100)
print(final_output)

FINAL APPLICATION OUTPUT
Safety and maintainability benefits associated with prognostics and health management include advanced warning of failure, elimination of redundant inspections, minimized unscheduled maintenance, and decreased test equipment requirement.

Retrieved sources:
1. NIST.AMS.100-18 — PDF page 17
2. NIST.AMS.100-2 — PDF page 7
3. NIST.AMS.100-2 — PDF page 10


## What you should remember

1. MiniLM performs retrieval; Qwen performs generation.
2. `chunk_embeddings @ query_vector` is the semantic search.
3. The evidence gate can stop an unsupported request before the LLM call.
4. The RAG prompt is our instruction/evidence structure; the chat template adds Qwen's model-specific role tokens.
5. `generator_model.generate()` is the actual LLM call.
6. Only newly generated token IDs are decoded for a causal language model.
7. Retrieved sources are appended by application code from stored metadata, supporting multiple pages without relying on the LLM.